In [1]:
import shap
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from rdkit.Chem import MolFromSmiles, rdFingerprintGenerator


import sys
sys.path.append('../')
import FragShapley

In [2]:
classification_datasets = ['O43570', 'P00915'][:1]
n_cv = 2
n_hyperopt = 2

n_explain = 5

expected_value = 'empty'

fpSize = 2048
radius = 2

random_state = 42

rfr_parameter_grid = {'n_estimators': [50]} # perform no hyperparameter optimization at the moment

results_folder = 'fingerprint_classification/'

In [3]:
rows_performance = []
df_expl = pd.DataFrame()

for classification_dataset in classification_datasets:
    print(f'On classification dataset: {classification_dataset}')

    # load dataset
    df = pd.read_csv(f'../0_datasets/classification/{classification_dataset}.csv')

    # cross validation
    cv = KFold(n_splits=n_cv,
               shuffle=True,
               random_state=random_state,
               )
    
    for split, (train_index, test_index) in enumerate(cv.split(df)):
        print(f'\tIn split: {split}')

        # split data and get fingerprints
        smiles_train = df.nonstereo_aromatic_smiles.iloc[train_index].to_list()
        smiles_test = df.nonstereo_aromatic_smiles.iloc[test_index].to_list()
        y_train, y_test = df.label.iloc[train_index].to_list(), df.label.iloc[test_index].to_list()

        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius,
                                                           fpSize=fpSize,
                                                           )
        fps_train = np.stack([mfpgen.GetFingerprintAsNumPy(MolFromSmiles(sm)) for sm in smiles_train])
        fps_test = np.stack([mfpgen.GetFingerprintAsNumPy(MolFromSmiles(sm)) for sm in smiles_test])

        # set up hyper parameter optimization using GridSearchCV
        rfr = RandomForestClassifier(random_state=random_state)
        inner_cv = KFold(n_splits=n_hyperopt,
                         shuffle=True,
                         random_state=random_state,
                         )
        gridCV = GridSearchCV(estimator=rfr,
                             param_grid=rfr_parameter_grid,
                             scoring='matthews_corrcoef', # use RMSE here
                             refit=True, # we want to use the best estimator afterwards
                             )
        gridCV.fit(X=fps_train,
                   y=y_train)
        
        best_regr = gridCV.best_estimator_
        y_pred = best_regr.predict(fps_test)
        y_pred_proba = best_regr.predict_proba(fps_test)[:, 1]
        rows_performance.append({'dataset': classification_dataset,
                                 'split': split,
                                 'best_params': gridCV.best_params_,
                                 'train_index': train_index,
                                 'test_index': test_index,
                                 'y_test': y_test,
                                 'y_pred': y_pred,
                                 'y_pred_proba': y_pred_proba,
                                 })

        # now the whole explanation part
        smiles_explain = smiles_test[:n_explain]

        # run FragShapley
        frag_explainer = FragShapley.FragmentExplainer(model = best_regr,
                                                          fingerprint_generator=mfpgen,
                                                          fragmentation_method='BRICS',
                                                          expected_value=expected_value)
        ev_frag = frag_explainer.expected_value
        ev_frag_logit = frag_explainer.expected_value_logit

        results_dicts, frag_to_atom_ids, atom_id_to_bits = frag_explainer.explain(smiles_explain, return_atom_id_to_bits=True)

        results = {'dataset': classification_dataset,
                   'split': split,
                   'smiles': smiles_explain,
                   'y_true': y_test[:n_explain],
                   'y_pred': y_pred[:n_explain],
                   'y_pred_proba': y_pred_proba[:n_explain],
                   'fragExplainer_result': results_dicts,
                   'fragExplainer_expected_value': [ev_frag for _ in smiles_explain],
                   'fragExplainer_expected_value_logits': [ev_frag_logit for _ in smiles_explain],
                   'frag_to_atom_ids': frag_to_atom_ids,
                   'atom_id_to_bits': atom_id_to_bits,
                   }
        df_expl_inner = pd.DataFrame(results)
        df_expl = pd.concat((df_expl, df_expl_inner))

df_performance = pd.DataFrame(rows_performance)
df_performance.to_pickle(results_folder + 'df_performance.pkl')
df_expl.to_pickle(results_folder + 'df_explanation.pkl')

On classification dataset: O43570
	In split: 0
	In split: 1


# Performance Analysis

In [4]:
df_performance = pd.read_pickle(results_folder + 'df_performance.pkl')
df_expl = pd.read_pickle(results_folder + 'df_explanation.pkl')

In [5]:
from sklearn.metrics import balanced_accuracy_score, f1_score, matthews_corrcoef

df_performance['BACC'] = df_performance.apply(lambda x: balanced_accuracy_score(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['F1'] = df_performance.apply(lambda x: f1_score(y_true=x.y_test, y_pred=x.y_pred), axis=1)
df_performance['MCC'] = df_performance.apply(lambda x: matthews_corrcoef(y_true=x.y_test, y_pred=x.y_pred), axis=1)

In [6]:
df_performance

,dataset,split,best_params,train_index,test_index,y_test,y_pred,y_pred_proba,BACC,F1,MCC
0,O43570,0,{'n_estimators': 50},"[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 16, 18...","[8, 12, 14, 15, 17, 19, 23, 24, 25, 26, 29, 30...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[1.0, 0.98, 0.94, 0.92, 0.94, 0.94, 0.74, 0.92...",0.964622,0.964809,0.929204
1,O43570,1,{'n_estimators': 50},"[8, 12, 14, 15, 17, 19, 23, 24, 25, 26, 29, 30...","[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 16, 18...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.86, 0.94, 0.8, 0.98, 1.0, 0.98, 0.92, 0.98,...",0.974616,0.974359,0.949243


In [7]:
def check_fragment_values_classification(fragment_dict, expected_value_logits, y_pred_proba, eps=1e-6):
    return np.allclose((sum(fragment_dict.values()) + expected_value_logits), np.log(1./(1. - y_pred_proba + eps)))

In [8]:
df_expl['check_fragExplainer_output'] = df_expl.apply(lambda x: check_fragment_values_classification(x.fragExplainer_result, x.fragExplainer_expected_value_logits, x.y_pred_proba), axis=1)
df_expl['check_fragExplainer_output'].value_counts()

check_fragExplainer_output
True    10
Name: count, dtype: int64

In [9]:
df_expl['sum_fragExplainer_ev'] = df_expl.apply(lambda x: sum(x.fragExplainer_result.values()) + x.fragExplainer_expected_value_logits, axis=1)
df_expl['logits_predict'] = df_expl.y_pred_proba.apply(lambda x: np.log(1./(1. - x + 1e-6)))

In [10]:
df_expl

,dataset,split,smiles,y_true,y_pred,y_pred_proba,fragExplainer_result,fragExplainer_expected_value,fragExplainer_expected_value_logits,frag_to_atom_ids,atom_id_to_bits,check_fragExplainer_output,sum_fragExplainer_ev,logits_predict
0,O43570,0,C#CCNc1ccc(S(N)(=O)=O)cc1,1,1,1.00,"{0: -0.2714410295362416, 1: -0.174363337440787...",0.82,1.714793,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...","{4: [62, 191, 1380], 2: [80, 1057, 1756], 7: [...",True,13.815511,13.815511
1,O43570,0,C#CCOP(=O)(Oc1ccc(S(N)(=O)=O)cc1)c1ccccc1,1,1,0.98,"{0: -3.497145381314736, 1: -3.732146570939296,...",0.82,1.714793,"{0: [0, 1, 2], 1: [3, 4, 5, 6, 17, 18, 19, 20,...","{6: [24, 335, 695], 4: [44, 192, 1586], 2: [80...",True,3.911973,3.911973
2,O43570,0,C#CCOc1cc(N(CC)CC)ccc1C=NCc1ccc(S(N)(=O)=O)cc1,1,1,0.94,"{0: -1.1944271069786974, 1: -0.499869418214349...",0.82,1.714793,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 12, 13, 14...","{2: [80, 1623, 1918], 8: [80, 180, 932], 10: [...",True,2.813394,2.813394
3,O43570,0,C#CCOc1ccc(Br)cc1C=NCc1ccc(S(N)(=O)=O)cc1,1,1,0.92,"{0: -1.0378145585387788, 1: -1.159778388860268...",0.82,1.714793,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...","{2: [80, 1623, 1918], 13: [80, 372, 1435], 5: ...",True,2.525716,2.525716
4,O43570,0,C#CCOc1ccc(C=NCc2ccc(S(N)(=O)=O)cc2)cc1OC,1,1,0.94,"{0: -1.049336827524296, 1: -3.9057106320874047...",0.82,1.714793,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6, 7, 8, 9, 1...","{6: [25, 1750, 1873], 2: [80, 1623, 1918], 10:...",True,2.813394,2.813394
0,O43570,1,Brc1ccc(-c2nn(-c3ccccc3)cc2-c2nc3ccccc3[nH]2)cc1,1,1,0.86,"{0: -0.396533306586084, 1: -0.0952628608919010...",0.90,2.302575,"{0: [0, 1, 2, 3, 4, 25, 26], 1: [5, 6, 7, 14, ...","{23: [119, 535, 1380], 24: [180, 1114, 1745], ...",True,1.966106,1.966106
1,O43570,1,C#CCCCOc1ccc2ccc(=O)oc2c1,1,1,0.94,"{0: -2.2674664928311414, 1: -2.411303536087767...",0.90,2.302575,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...","{4: [13, 80, 1873], 2: [80, 413, 448], 3: [80,...",True,2.813394,2.813394
2,O43570,1,C#CCCCOc1ccc2ccc(=S)oc2c1,1,1,0.80,"{0: -0.534801484928252, 1: -0.428255278080139,...",0.90,2.302575,"{0: [0, 1, 2, 3, 4], 1: [5], 2: [6, 7, 8, 9, 1...","{4: [13, 80, 1873], 2: [80, 413, 448], 3: [80,...",True,1.609433,1.609433
3,O43570,1,C#CCN(CC#C)CCCCOc1ccc(S(N)(=O)=O)cc1,1,1,0.98,"{0: -0.16724755216499834, 1: -0.07708179607705...",0.90,2.302575,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8,...","{10: [13, 80, 1873], 2: [80, 1317, 1350], 4: [...",True,3.911973,3.911973
4,O43570,1,C#CCN(CC#C)CCc1ccc(S(N)(=O)=O)cc1,1,1,1.00,"{0: -0.2106342912318404, 1: -0.085968193426926...",0.90,2.302575,"{0: [0, 1, 2], 1: [3], 2: [4, 5, 6], 3: [7, 8]...","{7: [26, 80, 1951], 2: [80, 1317, 1350], 4: [8...",True,13.815511,13.815511
